## Performance Profiling

Testing the performance of different components of the Experiment class

In [ ]:
import time

profiling_results = {}

print("="*80)
print("1. PROFILING EXPERIMENT INITIALIZATION")
print("="*80)

# Reset to initial parameters for consistent testing
gm = 0.03 * 2 * np.pi
inverse_pulse_width = 0.1 * gm

custom_constants = PhysicalConstants(
    chi = 0.5 * gm,
    photon_cavity_coupling = gm,
    inverse_pulse_width = 0.1 * gm
)

custom_dims = SystemDimensions(
    cavity_levels=2,
    qubit_levels=2,
    field_levels=2
)

custom_measurement = MeasurementProtocol(
    measurement_times = None,
    initial_time=-8.0/inverse_pulse_width,
    final_time=8.0/inverse_pulse_width,
    time_interval=8.0/inverse_pulse_width,
    initial_time_uncertainty=.0/inverse_pulse_width
)

initial_state = InitialStateConfig(
    state_type=InitialStateType.SINGLE_PHOTON
)

noise_config = NoiseConfiguration(
    depolarizing = 0.000,
    dephasing = 0.000,
    relaxation = 0.000
)

exp_parameters = ExperimentalParameters(
    physical_constants=custom_constants,
    system_dims=custom_dims,
    measurement=custom_measurement,
    initial_state=initial_state,
    noise_config=noise_config
)

# Time circuit creation
t0 = time.perf_counter()
initial_circuit = create_ry_circuit(n_qubits=1, theta_values=[np.pi/2])
final_circuit = create_ry_circuit(n_qubits=1, theta_values=[-np.pi/2])
t1 = time.perf_counter()
profiling_results["circuit_creation_ms"] = (t1 - t0) * 1000
print(f"\n✓ Circuit creation: {profiling_results['circuit_creation_ms']:.2f} ms")

# Time experiment initialization
t0 = time.perf_counter()
test_experiment = Experiment(
    experimental_params=exp_parameters,
    initial_circuit=initial_circuit,
    final_circuit=final_circuit,
)
t1 = time.perf_counter()
profiling_results["experiment_initialization_ms"] = (t1 - t0) * 1000
print(f"✓ Experiment initialization: {profiling_results['experiment_initialization_ms']:.2f} ms")

In [ ]:
print("\n" + "="*80)
print("2. PROFILING CIRCUIT UNITARY COMPUTATION")
print("="*80)

# Clear cache to test cold start
test_experiment._cached_circuit_unitaries = None

# Time first call (includes computation)
t0 = time.perf_counter()
unitaries = test_experiment._prepare_circuit_unitaries()
t1 = time.perf_counter()
first_unitary_ms = (t1 - t0) * 1000
profiling_results["unitary_prepare_first_ms"] = first_unitary_ms
print(f"\n✓ First call (compute + cache): {first_unitary_ms:.2f} ms")

# Time second call (from cache)
t0 = time.perf_counter()
unitaries = test_experiment._prepare_circuit_unitaries()
t1 = time.perf_counter()
second_unitary_ms = (t1 - t0) * 1000
profiling_results["unitary_prepare_cached_ms"] = second_unitary_ms
print(f"✓ Second call (from cache): {second_unitary_ms:.4f} ms")

if second_unitary_ms > 0.001:
    speedup = first_unitary_ms / second_unitary_ms
    print(f"  -> Caching speedup: {speedup:.1f}x")
else:
    print("  -> Caching speedup: effectively instantaneous")

In [ ]:
print("\n" + "="*80)
print("3. PROFILING SINGLE SIMULATION")
print("="*80)

# Time complete simulation
t0 = time.perf_counter()
results_test = test_experiment.run_simulation(batch_size=1)
t1 = time.perf_counter()
run_simulation_first_ms = (t1 - t0) * 1000
profiling_results["run_simulation_first_ms"] = run_simulation_first_ms
print(f"\n✓ Complete run_simulation: {run_simulation_first_ms:.2f} ms")

# Break down simulation components
print("\n--- Breaking down simulation components ---")

rho0 = test_experiment._cached_initial_state
solver_with = test_experiment.get_solver_with_interaction()
solver_without = test_experiment.get_solver_no_interaction()
measurement_times = test_experiment.experimental_params.get_measurement_times_with_uncertainty(1)
circuit_unitaries = test_experiment._prepare_circuit_unitaries()

# Time single evolution (with photon)
t0 = time.perf_counter()
prob_with = test_experiment.simulation(
    solver=solver_with,
    rho=rho0,
    measurements=measurement_times,
    precomputed_unitaries=circuit_unitaries,
)
t1 = time.perf_counter()
evolution_with_ms = (t1 - t0) * 1000
profiling_results["evolution_with_photon_ms"] = evolution_with_ms
print(f"  ✓ Evolution WITH photon: {evolution_with_ms:.2f} ms")

# Time single evolution (without photon)
t0 = time.perf_counter()
prob_without = test_experiment.simulation(
    solver=solver_without,
    rho=rho0,
    measurements=measurement_times,
    precomputed_unitaries=circuit_unitaries,
)
t1 = time.perf_counter()
evolution_without_ms = (t1 - t0) * 1000
profiling_results["evolution_without_photon_ms"] = evolution_without_ms
print(f"  ✓ Evolution WITHOUT photon: {evolution_without_ms:.2f} ms")

In [ ]:
print("\n" + "="*80)
print("4. PROFILING TIME EVOLUTION")
print("="*80)

time_evolution_ms = {}

# Test with different numbers of points
for n_points in [50, 100, 200, 300]:
    t0 = time.perf_counter()
    result_evol = test_experiment.time_evolution(n_points=n_points, with_interaction=True)
    t1 = time.perf_counter()
    elapsed_ms = (t1 - t0) * 1000
    time_evolution_ms[n_points] = elapsed_ms
    print(f"\n✓ Time evolution ({n_points:3d} points): {elapsed_ms:6.2f} ms ({elapsed_ms/n_points:5.2f} ms/point)")

profiling_results["time_evolution_ms"] = time_evolution_ms

In [ ]:
print("\n" + "="*80)
print("5. PROFILING OPTIMIZATION")
print("="*80)

# Test single optimization step
t0 = time.perf_counter()
history_test = test_experiment.optimize_rotations(
    initial_values=[1.3, -1.0],
    num_steps=10,
    verbose=False,
    batch_size=1,
    tolerance=1e-10,  # Won't converge, just test speed
    optimizer=optax.sgd(learning_rate=0.5)
)
t1 = time.perf_counter()

optimization_total_ms = (t1 - t0) * 1000
optimization_step_ms = optimization_total_ms / 10
profiling_results["optimization_10_steps_ms"] = optimization_total_ms
profiling_results["optimization_step_ms"] = optimization_step_ms
print(f"\n✓ 10 optimization steps: {optimization_total_ms:.2f} ms")
print(f"✓ Average per step: {optimization_step_ms:.2f} ms")

print("\n" + "="*80)
print("PROFILING SUMMARY")
print("="*80)
print("\nRun the next summary cell to generate metrics from the measured values.")

In [ ]:
print("\n" + "="*80)
print("6. PROFILING SECOND RUN (after JIT compilation)")
print("="*80)

# Run a second simulation to see if it's faster (should be after JAX compilation)
t0 = time.perf_counter()
results_test2 = test_experiment.run_simulation(batch_size=1)
t1 = time.perf_counter()
run_simulation_second_ms = (t1 - t0) * 1000
profiling_results["run_simulation_second_ms"] = run_simulation_second_ms
print(f"\n✓ Second run_simulation: {run_simulation_second_ms:.2f} ms")

first_run_ms = profiling_results.get("run_simulation_first_ms")
if first_run_ms is not None and run_simulation_second_ms > 0:
    speedup = first_run_ms / run_simulation_second_ms
    profiling_results["jit_speedup_factor"] = speedup
    print(f"  -> Speedup vs first run: {speedup:.1f}x faster")
else:
    print("  -> Speedup not available (run the first simulation profiling cell first).")

In [ ]:
print("\n" + "="*80)
print("7. TESTING FIXED CACHE (reload module first)")
print("="*80)

# Reload the module to get the fixed version
import importlib
import qsopt.core.experiment.experiment as exp_module
importlib.reload(exp_module)
from qsopt.core.experiment import Experiment

# Create new experiment with fixed code
test_experiment_fixed = Experiment(
    experimental_params=exp_parameters,
    initial_circuit=initial_circuit,
    final_circuit=final_circuit,
)

# Clear cache to test cold start
test_experiment_fixed._cached_circuit_unitaries = None

# Time first call (should compute + cache)
t0 = time.perf_counter()
unitaries = test_experiment_fixed._prepare_circuit_unitaries()
t1 = time.perf_counter()
print(f"\n✓ First call (compute + cache): {(t1-t0)*1000:.2f} ms")

# Time second call (should return from cache immediately)
t0 = time.perf_counter()
unitaries = test_experiment_fixed._prepare_circuit_unitaries()
t1 = time.perf_counter()
print(f"✓ Second call (from cache): {(t1-t0)*1000:.4f} ms")

if (t1-t0)*1000 < 0.1:
    print("  → ✓ CACHE WORKING! Near-instant return")
else:
    print(f"  → ⚠ Cache not working properly, still takes {(t1-t0)*1000:.2f}ms")

## Performance Summary & Recommendations

This section uses dynamically measured values from the profiling above.

Run the next code cell to generate an up-to-date summary from `profiling_results`.

In [ ]:
print("\n" + "="*80)
print("DYNAMIC PERFORMANCE SUMMARY")
print("="*80)

required_keys = [
    "run_simulation_first_ms",
    "run_simulation_second_ms",
    "evolution_with_photon_ms",
    "evolution_without_photon_ms",
    "optimization_step_ms",
]

missing_keys = [k for k in required_keys if k not in profiling_results]
if missing_keys:
    print("Missing measurements. Run cells 1 to 6 before this summary cell.")
    print(f"Missing keys: {missing_keys}")
else:
    first_ms = profiling_results["run_simulation_first_ms"]
    second_ms = profiling_results["run_simulation_second_ms"]
    with_ms = profiling_results["evolution_with_photon_ms"]
    without_ms = profiling_results["evolution_without_photon_ms"]
    opt_step_ms = profiling_results["optimization_step_ms"]
    jit_speedup = first_ms / second_ms if second_ms > 0 else float("inf")

    print(f"1. First-run JIT overhead: first run = {first_ms:.2f} ms")
    print(f"   Second run = {second_ms:.2f} ms ({jit_speedup:.1f}x faster)")

    print(f"2. Post-warmup simulation performance: {second_ms:.2f} ms per run")
    print(f"   - Evolution WITH photon: {with_ms:.2f} ms")
    print(f"   - Evolution WITHOUT photon: {without_ms:.2f} ms")

    evolution_data = profiling_results.get("time_evolution_ms", {})
    if evolution_data:
        print("3. Time evolution scaling:")
        for n_points in sorted(evolution_data):
            total_ms = evolution_data[n_points]
            print(f"   - {n_points:3d} points: {total_ms/n_points:.2f} ms/point")

    print(f"4. Optimization average cost: {opt_step_ms:.2f} ms per step")

print("\nRecommendations:")
print("1. The first run is expected to be slower because of JAX compilation.")
print("2. For benchmarking, always compare warm runs, not cold starts.")
print("3. Use medium/large grids for time-evolution to amortize setup costs.")
print("4. Keep caching enabled when sweeping parameters.")

In [ ]:
print("="*80)
print("FINAL VERIFICATION - Testing all optimizations")
print("="*80)

# Reload modules to get latest optimizations
import importlib
import qsopt.core.circuit as circuit_module
import qsopt.core.experiment.experiment as exp_module
importlib.reload(circuit_module)
importlib.reload(exp_module)
from qsopt.core.circuit import create_ry_circuit
from qsopt.core.experiment import Experiment

# Create fresh circuits and experiment
init_circ = create_ry_circuit(n_qubits=1, theta_values=[np.pi/2])
final_circ = create_ry_circuit(n_qubits=1, theta_values=[-np.pi/2])

optimized_exp = Experiment(
    experimental_params=exp_parameters,
    initial_circuit=init_circ,
    final_circuit=final_circ,
)

print("\n1. Testing circuit unitary caching:")
# Clear cache
init_circ._cached_unitary_jax = None
init_circ._cached_unitary_qutip = None

t0 = time.perf_counter()
u1 = init_circ.get_unitary(qutip=False)
t1 = time.perf_counter()
print(f"   First call: {(t1-t0)*1000:.3f} ms")

t0 = time.perf_counter()
u2 = init_circ.get_unitary(qutip=False)
t1 = time.perf_counter()
print(f"   Second call (cached): {(t1-t0)*1000:.3f} ms")
print(f"   → Speedup: {((t1-t0)*1000) < 0.1 and '✓ Cache working!' or '⚠ Still computing'}")

print("\n2. Testing experiment-level caching:")
optimized_exp._cached_circuit_unitaries = None

t0 = time.perf_counter()
u = optimized_exp._prepare_circuit_unitaries()
t1 = time.perf_counter()
print(f"   First call: {(t1-t0)*1000:.3f} ms")

t0 = time.perf_counter()
u = optimized_exp._prepare_circuit_unitaries()
t1 = time.perf_counter()
print(f"   Second call (cached): {(t1-t0)*1000:.3f} ms")
print(f"   → Speedup: {((t1-t0)*1000) < 0.1 and '✓ Cache working!' or '⚠ Still computing'}")

print("\n3. Testing full simulation performance:")
t0 = time.perf_counter()
result = optimized_exp.run_simulation(batch_size=1)
t1 = time.perf_counter()
print(f"   First run (with JIT): {(t1-t0)*1000:.1f} ms")

t0 = time.perf_counter()
result = optimized_exp.run_simulation(batch_size=1)
t1 = time.perf_counter()
print(f"   Second run (cached): {(t1-t0)*1000:.1f} ms")

print("\n" + "="*80)
print("✓ All optimizations verified!")
print("="*80)